In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from tqdm import tqdm
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [ ]:
P_eval = [
    "Given the input: '{input}', did the event '{cause}' cause '{effect}'?",
    "Taking the statement '{input}' into account, is '{cause}' the event that made '{effect}' occur?",
    "If '{input}' holds true, can we definitively state that '{effect}' was brought about by '{cause}'?",
    "Does '{cause}' directly explain why '{effect}' took place, based on the information: '{input}'?",
    "Judging from '{input}', is it correct to link the occurrence of '{effect}' back to '{cause}'?",
    "If we consider '{input}', does it logically suggest that '{cause}' is behind the event '{effect}'?",
    "Based on your understanding of '{input}', would you say that '{cause}' instigated '{effect}'?",
    "Reviewing '{input}', is it valid to claim '{cause}' was the catalyst for '{effect}'?",
    "Does the provided scenario '{input}' imply that the event '{effect}' was the outcome caused by '{cause}'?"
]

# function which applies P_eval to a unique input, and returns the paraphrase set
def get_formats(row):
    paraphrases = []
    for fmt in P_eval:
        paraphrases.append(fmt.format(input=row['sentence'], cause=row['cause'], effect=row['effect']))
    return paraphrases, row['label']

# torch Dataset class for DataLoader
class Data(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        paraphrases, label = get_formats(row) # returns input paraphrase set and associated labels at run time
        return paraphrases, label

# re-format batch. doesnt affect functionality, but needed to flatten the paraphrases
def fn(batch):
    paraphrases_list, labels = zip(*batch)
    return list(paraphrases_list), torch.tensor(labels)

eval_data_df = pd.read_csv('V_standard.csv') # doesnt matter which evaluation set is loaded, as long as 'sentence', 'cause' and 'effect' columns are present
eval_data = DataLoader(Data(eval_data_df), batch_size=64, collate_fn=fn)

In [ ]:
## LOAD MODEL

model_name = "google/flan-t5-large"
tokeniser = T5Tokenizer.from_pretrained(model_name)
model_loc = "path/to/ts1/weights"
model = T5ForConditionalGeneration.from_pretrained(model_name,
                                                   torch_dtype=torch.bfloat16,
                                                   device_map='cuda')

In [ ]:
ans_consistency_scores = []
rel_cos_consistency_scores = []
rel_dist_consistency_scores = []

# function which finds answer consistency for batch
def get_batch_ans_consis(flat_paraphrases, preds):
    num_paraphrases = len(P_eval)
    for group in range(0, len(flat_paraphrases), num_paraphrases): # iterate over each paraphrase set
        paraphrase_group_consis = sum(preds[group : group+num_paraphrases]) / num_paraphrases # find ans consistency of set
        if paraphrase_group_consis >= 0.5: # if more 1s than 0s predicted, append ans_consistency
            ans_consistency_scores.append(paraphrase_group_consis)
        else:
            ans_consistency_scores.append(1 - paraphrase_group_consis) # if more 0s than 1s predicted, append 1 - ans_consistency

# function which finds the cosine similarity and distance representational consistency metrics
def get_batch_rel_consis(decoder_output, flat_paraphrases):
    num_paraphrases = len(P_eval)
    for group in range(0, len(flat_paraphrases), num_paraphrases): # iterate over each paraphrase set
        decoder_output_group = decoder_output[group : group+num_paraphrases] # get representations of set
        mean_vec = torch.mean(decoder_output_group, dim=0) # find mean representation

        distances = torch.norm(decoder_output_group - mean_vec, dim=1).mean().item() # find mean of the distance to the mean representation
        sim = F.cosine_similarity(decoder_output_group, mean_vec, dim=1).mean().item() # find mean of the cosine similarities to mean representation

        rel_dist_consistency_scores.append(distances)
        rel_cos_consistency_scores.append(sim)

with torch.no_grad():
    for batch in tqdm(eval_data):
        all_paraphrases, labels = batch

        flat_paraphrases = [phrase for group in all_paraphrases for phrase in group] # each text will have 8 paraphrases, flatten these into 1D list
        inputs = tokeniser(flat_paraphrases, return_tensors='pt', truncation=True, padding=True, max_length=512).to('cuda') # tokenise flattened inputs

        #####################

        outputs = model.generate(**inputs, max_length=2, use_cache=False) # obtain outputs
        batch_predictions = [
                1 if "yes" == tokeniser.decode(output, skip_special_tokens=True).lower() else 0 for output in outputs # obtain predictions
            ]

        get_batch_ans_consis(flat_paraphrases, batch_predictions)

        ####################

        # obtain representations for the representational consistency metrics
        batch_size = inputs.input_ids.shape[0]
        decoder_input = torch.full((batch_size, 1), model.config.decoder_start_token_id, dtype=torch.long, device='cuda')
        outputs = model(input_ids=inputs.input_ids, decoder_input_ids=decoder_input, output_hidden_states=True)
        decoder_output = outputs.decoder_hidden_states[-1][:, -1, :]

        get_batch_rel_consis(decoder_output, flat_paraphrases)


# convert to numpy arrays
ans_consistency_scores = np.array(ans_consistency_scores)
rel_cos_consistency_scores = np.array(rel_cos_consistency_scores)
rel_dist_consistency_scores = np.array(rel_dist_consistency_scores)

# find mean of each metric
print(f"{rel_dist_consistency_scores.mean():.3f} & {rel_cos_consistency_scores.mean():.4f} & {ans_consistency_scores.mean()*100:.1f}% \\\\")